# 07 — Análise Anual do Custo de Manutenção por Carreta

**Fonte única de verdade (Single Source of Truth).** Este notebook reproduz, a partir da base consolidada
`data/raw/fato_wo_ml_2020-01-01_to_2025-12-31.csv` e do índice `reports/tables/04_cpi_fatores.csv`,
**todas** as estatísticas, tabelas e figuras utilizadas na apresentação (visão anual).

Nenhum gráfico, tabela ou estatística é elaborado manualmente para a apresentação: tudo é gerado aqui e
exportado para `reports/figures/anual/` (figuras) e `reports/tables/anual/` (tabelas).

## Abordagem metodológica
- **Variável resposta (Y):** custo interno **anual** de manutenção por carreta (CAD/ano), corrigido pela inflação
  canadense (CPI, Statistics Canada) e trazido para a data-base **dezembro/2025**.
- **Grão de análise:** carreta × ano.
- **Base:** consolidada em etapa anterior (não há reconstrução, joins ou feature engineering fora deste pipeline).
- **Sequência:** base consolidada → validação → correção monetária (CPI) → EDA (descritivas, distribuições,
  relação com Y, ranking, multicolinearidade) → seleção de variáveis → modelagem → avaliação.


In [1]:
import os, warnings, json
import numpy as np, pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

plt.rcParams.update({"figure.dpi":120,"font.size":10,"axes.grid":True,"grid.alpha":.25,
                     "axes.spines.top":False,"axes.spines.right":False})
AZ="#1f4e79"; LA="#5b9bd5"; CI="#c00000"

BASE = os.getcwd()
if not os.path.isdir(os.path.join(BASE,"dados")) and os.path.isdir(os.path.join(BASE,"..","dados")):
    BASE = os.path.abspath(os.path.join(BASE,".."))
RAW  = os.path.join(BASE,"dados","fato_wo_ml_2020-01-01_to_2025-12-31.csv")
CPIF = os.path.join(BASE,"dados","04_cpi_fatores.csv")
INTERIM = os.path.join(BASE,"dados")
FIG  = os.path.join(BASE,"figuras"); os.makedirs(FIG, exist_ok=True)
TAB  = os.path.join(BASE,"tabelas"); os.makedirs(TAB, exist_ok=True)

def savefig(fig,name):
    fig.tight_layout(); fig.savefig(os.path.join(FIG,name),bbox_inches="tight"); plt.close(fig)
def savetab(df,name):
    df.to_csv(os.path.join(TAB,name),index=False)
print("BASE:",BASE)


BASE: /Users/jeison.lima/Desktop/repos/quatro_norte/nova_versao_jeison


## 1. Base consolidada e correção monetária (CPI Canadá)

A empresa opera integralmente no Canadá; todos os custos são registrados em dólares canadenses (CAD).
Cada custo é multiplicado pelo fator do CPI *all-items* (StatCan) que o converte para dezembro/2025.
A deflação elimina o efeito da inflação e permite comparações temporais consistentes.


In [2]:
ANOS=[2020,2021,2022,2023,2024,2025]

cpi = pd.read_csv(CPIF); cpi["ano"]=cpi["ano_mes"].str[:4].astype(int)
CPI = cpi.groupby("ano")["fator_cpi_para_2025_12"].mean().to_dict()
print("Fatores CPI médios por ano (para dez/2025):")
for a in ANOS: print(f"  {a}: {CPI.get(a):.4f}")

df = pd.read_csv(RAW)
df["DATA_OS"]=pd.to_datetime(df["DATA_OS"],errors="coerce")
df=df.dropna(subset=["DATA_OS"])
df["ano"]=df["DATA_OS"].dt.year
df=df[df["ano"].isin(ANOS)].copy()
df["custo"]=pd.to_numeric(df["TOTAL_CUSTO_INTERNO"],errors="coerce").fillna(0.0)
df["custo_defl"]=df["custo"]*df["ano"].map(CPI).fillna(1.0)
vm=df["VMRS"].astype(str).str.upper()
df["prev"]=vm.str.contains("PREVENT")|vm.str.strip().eq("PM")
df["delta_km"]=pd.to_numeric(df["DELTA_KM_DESDE_ULTIMA_OS"],errors="coerce")
df["odo"]=pd.to_numeric(df["KM_ACUMULADO_DATA_OS"],errors="coerce")
df["ano_entrada"]=pd.to_datetime(df["DATA_ENTRADA_SERVICO"],errors="coerce").dt.year
print(f"\nOrdens de serviço no período: {len(df):,}")
print(f"Carretas distintas com OS: {df['ID_CARRETA'].nunique():,}")
print(f"Custo interno nominal: CAD {df['custo'].sum():,.0f} | deflacionado (dez/2025): CAD {df['custo_defl'].sum():,.0f}")


Fatores CPI médios por ano (para dez/2025):
  2020: 1.2048
  2021: 1.1654
  2022: 1.0913
  2023: 1.0503
  2024: 1.0259
  2025: 1.0050



Ordens de serviço no período: 223,587
Carretas distintas com OS: 9,859
Custo interno nominal: CAD 77,029,327 | deflacionado (dez/2025): CAD 82,293,610


## 2. Construção da base carreta × ano

Agregação das OS por carreta e ano, com atributos de cadastro e variáveis de histórico **defasadas**
(usam apenas informação de anos anteriores — anti-vazamento).


In [3]:
def moda(s):
    s=s.dropna(); return s.value_counts().index[0] if len(s) else np.nan
def vmrs_top(g):
    t=g.groupby("VMRS")["custo_defl"].sum(); return t.idxmax() if len(t) else np.nan

g=df.groupby(["ID_CARRETA","ano"])
base=g.agg(
    y_custo_anual=("custo_defl","sum"),
    custo_nominal=("custo","sum"),
    n_os_ano=("ID_OS","size"),
    n_os_preventivas_ano=("prev","sum"),
    custo_preventivo_ano=("custo_defl", lambda s: s[df.loc[s.index,"prev"]].sum()),
    n_sistemas_vmrs=("VMRS","nunique"),
    km_rodado_ano=("delta_km", lambda s: s[(s>0)&(s<20000)].sum()),
    delta_km_medio_os=("delta_km","mean"),
    km_acumulado_fim_ano=("odo","max"),
    provincia_operacao=("PROVINCIA_ESTADO", moda),
    ano_modelo=("ANO_MODELO","first"),
    ano_entrada=("ano_entrada","first"),
    eixos=("EIXOS","first"), comprimento=("COMPRIMENTO","first"),
    flag_refrigerado=("FLAG_REFRIGERADO","first"),
    tailgate_flag=("TAILGATE_FLAG","first"), unit_subtype=("UNIT_SUBTYPE","first"),
    tire_size=("TIRE_SIZE","first"), suspension_type=("SUSPENSION_TYPE","first"),
    new_used_indicator=("NEW_USED_INDICATOR","first"),
    cod_montadora=("COD_MONTADORA","first"),
).reset_index()
base["vmrs_predominante"]=g.apply(vmrs_top).values
base["idade_carreta"]=base["ano"]-base["ano_entrada"].fillna(base["ano_modelo"])
base["share_prev"]=base["custo_preventivo_ano"]/base["y_custo_anual"].replace(0,np.nan)
base["custo_medio_os"]=base["y_custo_anual"]/base["n_os_ano"]

# --- features de historico defasadas (anti-vazamento) ---
os_dates=df.groupby("ID_CARRETA")["DATA_OS"].apply(lambda s: sorted(s.tolist())).to_dict()
recs=[]
for cid,gc in base.groupby("ID_CARRETA"):
    gc=gc.sort_values("ano"); cum_c=0.0; cum_n=0; prev_c=np.nan; prev_n=np.nan; prev_odo=np.nan
    for _,r in gc.iterrows():
        ano=r["ano"]
        ds=[d for d in os_dates.get(cid,[]) if d < pd.Timestamp(int(ano),1,1)]
        interv=float(np.mean(np.diff([d.value for d in ds])/86400e9)) if len(ds)>=2 else np.nan
        anos_ult=((pd.Timestamp(int(ano),1,1)-ds[-1]).days/365.25) if ds else np.nan
        recs.append((cid,ano,cum_c,cum_n,prev_c,prev_n,interv,anos_ult,prev_odo))
        cum_c+=r["y_custo_anual"]; cum_n+=r["n_os_ano"]
        prev_c=r["y_custo_anual"]; prev_n=r["n_os_ano"]; prev_odo=r["km_acumulado_fim_ano"]
hist=pd.DataFrame(recs,columns=["ID_CARRETA","ano","custo_acum","n_os_acum","custo_ano_anterior",
    "n_os_ano_anterior","intervalo_medio_os_hist","anos_desde_ultima_os","km_acumulado_defasado"])
base=base.merge(hist,on=["ID_CARRETA","ano"])

base.to_csv(os.path.join(INTERIM,"base_anual_carreta_ano.csv"),index=False)
print(f"Base carreta-ano: {len(base):,} linhas | {base['ID_CARRETA'].nunique():,} carretas | anos {base['ano'].min()}-{base['ano'].max()}")
print("salvo: dados/base_anual_carreta_ano.csv")
base.head()


Base carreta-ano: 47,666 linhas | 9,859 carretas | anos 2020-2025
salvo: dados/base_anual_carreta_ano.csv


,ID_CARRETA,ano,y_custo_anual,custo_nominal,n_os_ano,n_os_preventivas_ano,custo_preventivo_ano,n_sistemas_vmrs,km_rodado_ano,delta_km_medio_os,...,idade_carreta,share_prev,custo_medio_os,custo_acum,n_os_acum,custo_ano_anterior,n_os_ano_anterior,intervalo_medio_os_hist,anos_desde_ultima_os,km_acumulado_defasado
0,19,2020,100.405243,83.34,1,1,100.405243,1,0.0,NaN,...,22.0,1.000000,100.405243,0.000000,0,NaN,NaN,NaN,NaN,NaN
1,19,2021,611.256988,524.50,2,1,150.687376,2,327.0,163.5,...,23.0,0.246520,305.628494,100.405243,1,100.405243,1.0,NaN,0.514716,51923.0
2,19,2022,101.276081,92.80,2,2,101.276081,1,680.0,340.0,...,24.0,1.000000,50.638040,711.662232,3,611.256988,2.0,205.415243,0.388775,52250.0
3,19,2023,211.886727,201.73,2,1,192.245217,2,81.0,40.5,...,25.0,0.907302,105.943364,812.938313,5,101.276081,2.0,183.453863,0.503765,52930.0
4,19,2024,171.430999,167.11,2,0,0.000000,2,1175.0,587.5,...,26.0,0.000000,85.715500,1024.825040,7,211.886727,2.0,181.909716,0.522930,53011.0


## 3. Variáveis candidatas

Todas as variáveis abaixo entram como **candidatas**. Nenhuma é tratada como importante antes da EDA.


In [4]:
NUM=["ano_modelo","idade_carreta","eixos","comprimento","km_rodado_ano","delta_km_medio_os",
     "km_acumulado_fim_ano","km_acumulado_defasado","n_os_ano","n_os_preventivas_ano",
     "custo_preventivo_ano","share_prev","n_sistemas_vmrs","custo_medio_os",
     "custo_acum","n_os_acum","custo_ano_anterior","n_os_ano_anterior",
     "intervalo_medio_os_hist","anos_desde_ultima_os"]
CAT=["flag_refrigerado","tailgate_flag","unit_subtype","tire_size","suspension_type",
     "new_used_indicator","cod_montadora","provincia_operacao","vmrs_predominante"]
# usa apenas colunas realmente presentes na base
NUM=[c for c in NUM if c in base.columns]
CAT=[c for c in CAT if c in base.columns]
y=base["y_custo_anual"]
print(f"Candidatas: {len(NUM)} numéricas + {len(CAT)} categóricas = {len(NUM)+len(CAT)}")


Candidatas: 20 numéricas + 9 categóricas = 29


## 4. Estatística descritiva

Para cada variável numérica: N, ausentes, média, mediana, desvio-padrão, coeficiente de variação,
quartis, mínimo, máximo, assimetria e curtose. Para categóricas: frequências absoluta/relativa e Y médio.


In [5]:
rows=[]
sy=y.dropna()
rows.append(dict(variavel="Y_custo_anual",N=len(sy),ausentes=0,media=sy.mean(),mediana=sy.median(),
    desvio_padrao=sy.std(),coef_variacao=sy.std()/sy.mean(),Q1=sy.quantile(.25),Q3=sy.quantile(.75),
    minimo=sy.min(),maximo=sy.max(),assimetria=sy.skew(),curtose=sy.kurt()))
for f in NUM:
    s=base[f].dropna()
    rows.append(dict(variavel=f,N=len(s),ausentes=int(base[f].isna().sum()),media=s.mean(),mediana=s.median(),
        desvio_padrao=s.std(),coef_variacao=(s.std()/s.mean() if s.mean() else np.nan),
        Q1=s.quantile(.25),Q3=s.quantile(.75),minimo=s.min(),maximo=s.max(),
        assimetria=s.skew(),curtose=s.kurt()))
desc=pd.DataFrame(rows).round(3)
savetab(desc,"anual_descritivas_numericas.csv")

cat_rows=[]
for f in CAT:
    vc=base[f].value_counts(dropna=False)
    for k,v in vc.head(15).items():
        cat_rows.append(dict(variavel=f,categoria=str(k),freq_abs=int(v),freq_rel=round(v/len(base),4),
            y_medio=round(base.loc[base[f].astype(str)==str(k),"y_custo_anual"].mean(),2)))
savetab(pd.DataFrame(cat_rows),"anual_descritivas_categoricas.csv")
print("tabelas: anual_descritivas_numericas.csv, anual_descritivas_categoricas.csv")
desc


tabelas: anual_descritivas_numericas.csv, anual_descritivas_categoricas.csv


,variavel,N,ausentes,media,mediana,desvio_padrao,coef_variacao,Q1,Q3,minimo,maximo,assimetria,curtose
0,Y_custo_anual,47666,0,1726.464,859.498,2413.835,1.398,348.530,2075.767,-2899.070,62365.614,3.762,27.440
1,ano_modelo,47660,6,2014.655,2015.000,5.791,0.003,2011.000,2019.000,1982.000,2026.000,-0.441,-0.488
2,idade_carreta,47660,6,8.265,7.000,5.744,0.695,4.000,12.000,-2.000,39.000,0.494,-0.444
3,eixos,47579,87,2.077,2.000,0.275,0.132,2.000,2.000,1.000,4.000,3.249,10.140
4,comprimento,47031,635,52.238,53.000,3.736,0.072,53.000,53.000,28.000,60.000,-4.953,24.313
5,km_rodado_ano,47666,0,13796.645,8501.500,16431.657,1.191,1421.000,19094.000,0.000,149282.000,1.967,4.881
6,delta_km_medio_os,45222,2444,6069.416,3579.000,13627.030,2.245,1418.000,6830.875,-508719.333,1014231.500,19.710,1020.951
7,km_acumulado_fim_ano,47433,233,178557.673,111710.000,198636.818,1.112,40513.000,242587.000,-78620.000,3479389.000,2.187,8.523
8,km_acumulado_defasado,37600,10066,170696.998,104176.000,193373.116,1.133,37627.250,229912.000,-78620.000,2051521.000,2.106,5.998
9,n_os_ano,47666,0,4.691,3.000,4.268,0.910,2.000,6.000,1.000,59.000,2.445,9.070


## 5. Distribuição das variáveis (histogramas e boxplots)

In [6]:
cap=y.quantile(.99)
# Y: histograma, log, boxplot, evolução
fig,ax=plt.subplots(figsize=(7,4))
ax.hist(y[(y>=0)&(y<=cap)],bins=60,color=LA,edgecolor="white")
ax.axvline(y.median(),color=CI,ls="--",label=f"mediana {y.median():.0f}")
ax.axvline(y.mean(),color=AZ,ls="-",label=f"média {y.mean():.0f}")
ax.set(title="Distribuição do custo anual por carreta (CAD/ano, deflacionado)",xlabel="CAD/ano (até p99)",ylabel="carreta-ano"); ax.legend()
savefig(fig,"01_hist_y.png")

fig,ax=plt.subplots(figsize=(7,4)); yp=y[y>0]
ax.hist(np.log10(yp),bins=60,color=AZ,edgecolor="white")
ax.set(title="Custo anual por carreta em escala log10",xlabel="log10(CAD/ano)",ylabel="carreta-ano")
savefig(fig,"02_hist_y_log.png")

fig,ax=plt.subplots(figsize=(6,3.2))
ax.boxplot(y[(y>=0)&(y<=cap)],vert=False,showfliers=False,patch_artist=True,
           boxprops=dict(facecolor=LA),medianprops=dict(color=CI))
ax.set(title="Boxplot do custo anual por carreta (sem outliers extremos)",xlabel="CAD/ano"); ax.set_yticks([])
savefig(fig,"03_box_y.png")

ev=base.groupby("ano")["y_custo_anual"].agg(n="size",media="mean",mediana="median",
        p90=lambda s:s.quantile(.9)).round(2)
ev.reset_index().to_csv(os.path.join(TAB,"anual_evolucao_y.csv"),index=False)
fig,ax=plt.subplots(figsize=(7,4))
ax.plot(ev.index,ev["media"],"-o",color=AZ,label="média")
ax.plot(ev.index,ev["mediana"],"-o",color=CI,label="mediana")
ax.set(title="Evolução do custo anual por carreta (valores reais dez/2025)",xlabel="ano",ylabel="CAD/ano"); ax.legend()
savefig(fig,"04_evolucao_y.png")

# grade de histogramas das numéricas
fig,axs=plt.subplots(4,5,figsize=(16,11))
for ax,f in zip(axs.ravel(),NUM):
    s=base[f].dropna(); lo,hi=s.quantile(.01),s.quantile(.99)
    ax.hist(s[(s>=lo)&(s<=hi)],bins=40,color=LA,edgecolor="white"); ax.set_title(f,fontsize=8); ax.tick_params(labelsize=6)
for ax in axs.ravel()[len(NUM):]: ax.axis("off")
fig.suptitle("Distribuição das variáveis numéricas candidatas (1–99%)",fontsize=13)
savefig(fig,"12_grade_histogramas.png")
print("figuras Y + grade de histogramas geradas")


figuras Y + grade de histogramas geradas


In [7]:
# uma figura por variável numérica: histograma + boxplot (reproduz os slides por variável)
for f in NUM:
    s=base[f].dropna(); lo,hi=s.quantile(.01),s.quantile(.99); sc=s[(s>=lo)&(s<=hi)]
    fig,ax=plt.subplots(1,2,figsize=(9,3.4))
    ax[0].hist(sc,bins=40,color=LA,edgecolor="white"); ax[0].set_title(f"Histograma — {f}",fontsize=9)
    ax[1].boxplot(sc,vert=False,showfliers=False,patch_artist=True,boxprops=dict(facecolor=LA),medianprops=dict(color=CI))
    ax[1].set_title(f"Boxplot — {f}",fontsize=9); ax[1].set_yticks([])
    savefig(fig,f"num_{f}.png")
print(f"{len(NUM)} figuras por variável numérica em reports/figures/anual/num_*.png")


20 figuras por variável numérica em reports/figures/anual/num_*.png


## 6. Relação entre cada variável e Y

Numéricas: correlações de **Pearson** e **Spearman**. Categóricas: **η** (eta) e **ANOVA** (F, p-valor).


In [8]:
# numéricas x Y
cor=[]
for f in NUM:
    s=base[[f,"y_custo_anual"]].dropna()
    if len(s)<50: continue
    cor.append((f,s[f].corr(s["y_custo_anual"]),s[f].corr(s["y_custo_anual"],method="spearman"),len(s)))
cor_df=pd.DataFrame(cor,columns=["variavel","pearson","spearman","n"]).sort_values("spearman",key=lambda c:c.abs(),ascending=False)
savetab(cor_df.round(4),"anual_correlacao_y.csv")

# categóricas x Y (eta + ANOVA)
eta=[]
for f in CAT:
    s=base[[f,"y_custo_anual"]].dropna()
    if s[f].nunique()<2: continue
    grand=s["y_custo_anual"].mean(); ss_tot=((s["y_custo_anual"]-grand)**2).sum()
    ss_bet=s.groupby(f)["y_custo_anual"].apply(lambda x:len(x)*(x.mean()-grand)**2).sum()
    e2=ss_bet/ss_tot if ss_tot>0 else 0
    grupos=[x.values for _,x in s.groupby(f)["y_custo_anual"] if len(x)>1]
    try: F,pv=stats.f_oneway(*grupos)
    except: F,pv=(np.nan,np.nan)
    eta.append((f,e2**.5,e2,F,pv,s[f].nunique()))
eta_df=pd.DataFrame(eta,columns=["variavel","eta","eta2","F_anova","p_valor","n_categorias"]).sort_values("eta",ascending=False)
savetab(eta_df.round(4),"anual_eta_categoricas.csv")

# heatmap de correlação (Spearman) numéricas + Y
cols=["y_custo_anual"]+NUM; cm=base[cols].corr(method="spearman")
fig,ax=plt.subplots(figsize=(11,9)); im=ax.imshow(cm,cmap="RdBu_r",vmin=-1,vmax=1)
ax.set_xticks(range(len(cols))); ax.set_xticklabels(cols,rotation=90,fontsize=7)
ax.set_yticks(range(len(cols))); ax.set_yticklabels(cols,fontsize=7)
for i in range(len(cols)):
    for j in range(len(cols)):
        ax.text(j,i,f"{cm.iloc[i,j]:.2f}",ha="center",va="center",fontsize=5.5,
                color="white" if abs(cm.iloc[i,j])>.5 else "black")
fig.colorbar(im,fraction=.046,pad=.04); ax.set_title("Matriz de correlação de Spearman (numéricas × Y)")
savefig(fig,"05_heatmap_correlacao.png")

# scatters chave
for name,var,lab in [("10_scatter_n_os","n_os_ano","nº OS no ano"),("11_scatter_km","km_rodado_ano","km rodados no ano")]:
    if var not in base.columns: continue
    s=base[[var,"y_custo_anual"]].dropna(); s=s[s["y_custo_anual"]<=cap]
    fig,ax=plt.subplots(figsize=(6.5,4.5)); ax.scatter(s[var],s["y_custo_anual"],s=4,alpha=.12,color=AZ)
    ax.set(title=f"Y × {lab} (ρ={base[var].corr(base['y_custo_anual'],method='spearman'):.2f})",xlabel=lab,ylabel="CAD/ano")
    savefig(fig,f"{name}.png")

# boxplot de Y por cada categórica (top categorias)
for f in CAT:
    top=base[f].value_counts().head(8).index.tolist()
    if len(top)<2: continue
    grp=[base.loc[base[f].astype(str)==str(k),"y_custo_anual"].clip(upper=cap) for k in top]
    fig,ax=plt.subplots(figsize=(9,4.2))
    ax.boxplot(grp,labels=[str(k)[:14] for k in top],showfliers=False,patch_artist=True,
               boxprops=dict(facecolor=LA),medianprops=dict(color=CI))
    ax.set(title=f"Custo anual por carreta × {f} (top {len(top)})",ylabel="CAD/ano"); plt.xticks(rotation=30,ha="right")
    savefig(fig,f"cat_{f}.png")
print("relação com Y: tabelas + heatmap + scatters + boxplots por categoria")
cor_df.head(10)


relação com Y: tabelas + heatmap + scatters + boxplots por categoria


,variavel,pearson,spearman,n
13,custo_medio_os,0.509477,0.763560,47666
8,n_os_ano,0.746196,0.745303,47666
12,n_sistemas_vmrs,0.625687,0.672755,47666
16,custo_ano_anterior,0.584825,0.530949,37807
17,n_os_ano_anterior,0.585155,0.520901,37807
4,km_rodado_ano,0.490211,0.498169,47666
14,custo_acum,0.548700,0.469977,47666
15,n_os_acum,0.534952,0.466332,47666
6,km_acumulado_fim_ano,0.381085,0.443660,47433
18,intervalo_medio_os_hist,-0.318513,-0.428778,36026


## 7. Ranking das variáveis e multicolinearidade (VIF)

O ranking considera a força de associação (Spearman/η). A multicolinearidade (VIF) sinaliza redundância:
de cada par redundante mantém-se um representante na modelagem.


In [9]:
# VIF (numéricas)
X=base[NUM].dropna(); vifs=[]
for f in NUM:
    others=[c for c in NUM if c!=f]; A=X[others].values; b=X[f].values
    A1=np.column_stack([np.ones(len(A)),A]); coef,_,_,_=np.linalg.lstsq(A1,b,rcond=None); pred=A1@coef
    r2=1-((b-pred)**2).sum()/((b-b.mean())**2).sum(); vifs.append((f,1/(1-r2) if r2<1 else np.inf))
vif_df=pd.DataFrame(vifs,columns=["variavel","VIF"]).sort_values("VIF",ascending=False)
savetab(vif_df.round(3),"anual_vif.csv")

# ranking combinado
rank=cor_df.merge(vif_df,on="variavel",how="left"); rank["abs_spearman"]=rank["spearman"].abs()
rank=rank.sort_values("abs_spearman",ascending=False)
savetab(rank[["variavel","spearman","pearson","VIF","n"]].round(4),"anual_ranking_variaveis.csv")

# figura ranking (numéricas + categóricas em uma escala de associação)
assoc=pd.concat([
    cor_df.assign(forca=cor_df["spearman"].abs(),tipo="numérica")[["variavel","forca","tipo"]],
    eta_df.assign(forca=eta_df["eta"],tipo="categórica")[["variavel","forca","tipo"]]
]).sort_values("forca")
fig,ax=plt.subplots(figsize=(8,7))
ax.barh(assoc["variavel"],assoc["forca"],color=[AZ if t=="numérica" else LA for t in assoc["tipo"]])
ax.set(title="Ranking de associação com o custo anual (|Spearman| e η)",xlabel="força da associação (0–1)")
savefig(fig,"06_ranking_associacao.png")

# figura VIF
fig,ax=plt.subplots(figsize=(7.5,6)); vv=vif_df.sort_values("VIF")
ax.barh(vv["variavel"],vv["VIF"].clip(upper=100),color=[CI if v>=10 else LA for v in vv["VIF"]])
ax.axvline(10,color="black",ls="--",lw=.8,label="VIF=10")
ax.set(title="Multicolinearidade (VIF, teto visual 100)",xlabel="VIF"); ax.legend()
savefig(fig,"13_vif.png")
print("ranking + VIF: tabelas e figuras geradas")
rank.head(12)


ranking + VIF: tabelas e figuras geradas


,variavel,pearson,spearman,n,VIF,abs_spearman
0,custo_medio_os,0.509477,0.763560,47666,1.539057,0.763560
1,n_os_ano,0.746196,0.745303,47666,5.198262,0.745303
2,n_sistemas_vmrs,0.625687,0.672755,47666,3.072626,0.672755
3,custo_ano_anterior,0.584825,0.530949,37807,4.683866,0.530949
4,n_os_ano_anterior,0.585155,0.520901,37807,5.817377,0.520901
5,km_rodado_ano,0.490211,0.498169,47666,2.503677,0.498169
6,custo_acum,0.548700,0.469977,47666,8.684616,0.469977
7,n_os_acum,0.534952,0.466332,47666,9.076093,0.466332
8,km_acumulado_fim_ano,0.381085,0.443660,47433,93.593508,0.443660
9,intervalo_medio_os_hist,-0.318513,-0.428778,36026,1.448276,0.428778


## 8. Seleção das variáveis do modelo

O objetivo do projeto é **prever/estimar** o custo anual de uma carreta a partir de suas características.
Uma variável só pode entrar no modelo se estiver disponível **antes** do ano que se quer prever. Por isso a
seleção aplica um critério anti-vazamento em duas camadas:

- **Vazamento aritmético** — variáveis que são função do próprio Y: `custo_medio_os` (= Y/nº OS),
  `custo_preventivo_ano`, `share_prev`, `custo_nominal`.
- **Vazamento temporal (contemporâneo)** — variáveis medidas *no próprio ano*, que não são conhecidas no
  início dele e se movem mecanicamente com o custo: `n_os_ano`, `n_os_preventivas_ano`, `n_sistemas_vmrs`,
  `km_rodado_ano`, `delta_km_medio_os`, `km_acumulado_fim_ano`, `vmrs_predominante` (sistema de maior custo do ano).

Restam como preditores legítimos os **atributos do ativo** (idade, eixos, comprimento, tipo, região…) e o
**histórico defasado** (custo/OS/km de anos anteriores, recência). De `ano_modelo` × `idade_carreta`
(colineares) mantém-se `idade_carreta`.


In [10]:
LEAK_ARIT=["custo_medio_os","custo_preventivo_ano","share_prev","custo_nominal","y_custo_anual"]
LEAK_CONTEMP=["n_os_ano","n_os_preventivas_ano","n_sistemas_vmrs","km_rodado_ano",
              "delta_km_medio_os","km_acumulado_fim_ano","vmrs_predominante"]
DROP_COLIN=["ano_modelo"]
EXCL=set(LEAK_ARIT)|set(LEAK_CONTEMP)|set(DROP_COLIN)

NUM_PRED=[c for c in NUM if c not in EXCL]
CAT_PRED=[c for c in CAT if c not in EXCL]
# conjunto explicativo (com contemporâneas, apenas para sensibilidade — NÃO é preditivo)
NUM_EXPL=[c for c in NUM if c not in (set(LEAK_ARIT)|set(DROP_COLIN))]
CAT_EXPL=[c for c in CAT]

def motivo(v):
    if v in LEAK_ARIT:    return "excluída — derivada de Y (vazamento aritmético)"
    if v in LEAK_CONTEMP: return "excluída — medida no próprio ano (vazamento temporal)"
    if v in DROP_COLIN:   return "excluída — colinearidade (VIF, redundante com idade_carreta)"
    return "selecionada"

allvars=NUM+CAT
sel=pd.DataFrame({"variavel":allvars,
                  "tipo":["numérica"]*len(NUM)+["categórica"]*len(CAT),
                  "decisao":[motivo(v) for v in allvars]})
savetab(sel[sel.decisao=="selecionada"],"anual_variaveis_selecionadas.csv")
savetab(sel[sel.decisao!="selecionada"][["variavel","decisao"]],"anual_variaveis_excluidas.csv")
savetab(sel,"anual_selecao_completa.csv")
print("Preditoras selecionadas:",len(NUM_PRED)+len(CAT_PRED),
      "| Excluídas:",len(sel)-(len(NUM_PRED)+len(CAT_PRED)))
sel


Preditoras selecionadas: 18 | Excluídas: 11


,variavel,tipo,decisao
0,ano_modelo,numérica,"excluída — colinearidade (VIF, redundante com ..."
1,idade_carreta,numérica,selecionada
2,eixos,numérica,selecionada
3,comprimento,numérica,selecionada
4,km_rodado_ano,numérica,excluída — medida no próprio ano (vazamento te...
5,delta_km_medio_os,numérica,excluída — medida no próprio ano (vazamento te...
6,km_acumulado_fim_ano,numérica,excluída — medida no próprio ano (vazamento te...
7,km_acumulado_defasado,numérica,selecionada
8,n_os_ano,numérica,excluída — medida no próprio ano (vazamento te...
9,n_os_preventivas_ano,numérica,excluída — medida no próprio ano (vazamento te...


## 9. Modelagem

Split **temporal**: treino 2020–2024, teste 2025 (avaliação fora do tempo). O modelo **preditivo** usa apenas
as variáveis selecionadas (atributos do ativo + histórico defasado), refletindo o que se conhece antes do ano.
Compara-se regressão linear (log1p do Y), Random Forest e Gradient Boosting, com R², RMSE, MAE e WAPE.

Também é reportado, **apenas como sensibilidade explicativa** (não preditiva), um modelo que inclui as
variáveis contemporâneas: ele atinge R² maior porque incorpora `n_os_ano` e afins, que são quase o próprio Y —
serve para medir o teto de ajuste, não para prever.


In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.preprocessing import OrdinalEncoder, StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

bm=base.copy(); bm["y"]=bm["y_custo_anual"].clip(lower=0)
for c in set(CAT_PRED)|set(CAT_EXPL): bm[c]=bm[c].astype(str).fillna("NA")
tr=bm[bm["ano"]<=2024]; te=bm[bm["ano"]==2025]
ytr,yte=tr["y"],te["y"]
print(f"treino={len(tr):,} (2020-2024) | teste={len(te):,} (2025)")
print(f"preditoras: {len(NUM_PRED)} num + {len(CAT_PRED)} cat")

def metricas(yv,pv):
    return dict(R2=r2_score(yv,pv),RMSE=mean_squared_error(yv,pv)**.5,
                MAE=mean_absolute_error(yv,pv),WAPE=np.abs(yv-pv).sum()/np.abs(yv).sum())

pre_num=Pipeline([("imp",SimpleImputer(strategy="median")),("sc",StandardScaler())])
pre_cat_oh=Pipeline([("imp",SimpleImputer(strategy="constant",fill_value="NA")),
                     ("oh",OneHotEncoder(handle_unknown="ignore",max_categories=12,min_frequency=50))])
pre_cat_ord=Pipeline([("imp",SimpleImputer(strategy="constant",fill_value="NA")),
                      ("ord",OrdinalEncoder(handle_unknown="use_encoded_value",unknown_value=-1))])
results={}; preds={}

def fit_lin(NUMc,CATc,Xtr,Xte):
    ct=ColumnTransformer([("n",pre_num,NUMc),("c",pre_cat_oh,CATc)])
    m=Pipeline([("ct",ct),("m",LinearRegression())]); m.fit(Xtr,np.log1p(ytr))
    return m,np.expm1(m.predict(Xte)).clip(0,ytr.max())
def fit_tree(model,NUMc,CATc,Xtr,Xte):
    ct=ColumnTransformer([("n",pre_num,NUMc),("c",pre_cat_ord,CATc)])
    m=Pipeline([("ct",ct),("m",model)]); m.fit(Xtr,ytr)
    return m,m.predict(Xte).clip(min=0)

# ---- MODELOS PREDITIVOS (sem vazamento) ----
Xtr,Xte=tr[NUM_PRED+CAT_PRED],te[NUM_PRED+CAT_PRED]
lin,p_lin=fit_lin(NUM_PRED,CAT_PRED,Xtr,Xte)
results["Regressão Linear (log)"]=metricas(yte,p_lin); preds["Regressão Linear (log)"]=p_lin
rf,p_rf=fit_tree(RandomForestRegressor(n_estimators=300,min_samples_leaf=5,n_jobs=-1,random_state=42),NUM_PRED,CAT_PRED,Xtr,Xte)
results["Random Forest"]=metricas(yte,p_rf); preds["Random Forest"]=p_rf
gb,p_gb=fit_tree(HistGradientBoostingRegressor(max_iter=400,learning_rate=0.06,l2_regularization=1.0,random_state=42),NUM_PRED,CAT_PRED,Xtr,Xte)
results["Gradient Boosting"]=metricas(yte,p_gb); preds["Gradient Boosting"]=p_gb

# ---- SENSIBILIDADE EXPLICATIVA (com variáveis do ano — NÃO preditivo) ----
Xtr_e,Xte_e=tr[NUM_EXPL+CAT_EXPL],te[NUM_EXPL+CAT_EXPL]
gb_e,p_gbe=fit_tree(HistGradientBoostingRegressor(max_iter=400,learning_rate=0.06,l2_regularization=1.0,random_state=42),NUM_EXPL,CAT_EXPL,Xtr_e,Xte_e)
results["Gradient Boosting (explicativo · com variáveis do ano)"]=metricas(yte,p_gbe)

met=pd.DataFrame(results).T.reset_index().rename(columns={"index":"modelo"}).round({"R2":3,"RMSE":1,"MAE":1,"WAPE":3})
savetab(met,"anual_metricas_modelos.csv")
print(met.to_string(index=False))


treino=38,674 (2020-2024) | teste=8,992 (2025)
preditoras: 10 num + 8 cat


                                                modelo    R2   RMSE    MAE  WAPE
                                Regressão Linear (log) 0.082 2562.3 1253.2 0.620
                                         Random Forest 0.498 1894.3 1149.1 0.568
                                     Gradient Boosting 0.484 1921.1 1129.5 0.559
Gradient Boosting (explicativo · com variáveis do ano) 0.687 1496.9  836.6 0.414


In [12]:
# melhor entre os PREDITIVOS (a sensibilidade explicativa não concorre a "melhor")
PRED_MODELS={"Regressão Linear (log)":(lin,preds["Regressão Linear (log)"]),
             "Random Forest":(rf,preds["Random Forest"]),
             "Gradient Boosting":(gb,preds["Gradient Boosting"])}
met_pred=met[met["modelo"].isin(PRED_MODELS)]
best=met_pred.sort_values("R2",ascending=False).iloc[0]["modelo"]
best_model,pbest=PRED_MODELS[best]
print("Melhor modelo preditivo:",best)

# comparação (explicativo destacado como fora da comparação preditiva)
is_expl=met["modelo"].str.contains("explicativo")
cols=[("#c9a227" if e else AZ) for e in is_expl]
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
ax[0].bar(range(len(met)),met["R2"],color=cols); ax[0].set_title("R² (teste 2025)")
ax[0].set_xticks(range(len(met))); ax[0].set_xticklabels(met["modelo"],rotation=25,ha="right",fontsize=7)
ax[0].axhline(0,color="black",lw=.6)
ax[1].bar(range(len(met)),met["WAPE"],color=cols); ax[1].set_title("WAPE (menor = melhor)")
ax[1].set_xticks(range(len(met))); ax[1].set_xticklabels(met["modelo"],rotation=25,ha="right",fontsize=7)
fig.suptitle("Preditivo (azul) × sensibilidade explicativa com variáveis do ano (dourado)",fontsize=10)
savefig(fig,"14_comparacao_modelos.png")

# importância no melhor modelo PREDITIVO
pi=permutation_importance(best_model,Xte,yte,n_repeats=8,random_state=42,n_jobs=-1,scoring="r2")
imp=pd.DataFrame({"variavel":NUM_PRED+CAT_PRED,"importancia":pi.importances_mean,"desvio":pi.importances_std}).sort_values("importancia",ascending=False)
savetab(imp.round(4),"anual_importancia_variaveis.csv")
fig,ax=plt.subplots(figsize=(8,6)); ii=imp.head(15).sort_values("importancia")
ax.barh(ii["variavel"],ii["importancia"],xerr=ii["desvio"],color=AZ)
ax.set(title=f"Importância das variáveis (permutation · {best}, preditivo)",xlabel="queda no R²")
savefig(fig,"15_importancia_variaveis.png")

capp=np.quantile(yte,.99)
fig,ax=plt.subplots(figsize=(5.5,5.5)); ax.scatter(yte,pbest,s=5,alpha=.15,color=AZ); lim=[0,capp]
ax.plot(lim,lim,color=CI,ls="--"); ax.set(xlim=lim,ylim=lim,xlabel="observado (CAD/ano)",ylabel="previsto",title=f"Previsto × Observado — {best} (2025)")
savefig(fig,"16_previsto_vs_observado.png")
print("modelagem: métricas, importância e previsto×observado gerados")
imp.head(12)


Melhor modelo preditivo: Random Forest


modelagem: métricas, importância e previsto×observado gerados


,variavel,importancia,desvio
12,unit_subtype,0.094425,0.004382
4,custo_acum,0.089014,0.003031
0,idade_carreta,0.048099,0.001417
3,km_acumulado_defasado,0.031055,0.002296
2,comprimento,0.025245,0.000838
7,n_os_ano_anterior,0.024002,0.001781
17,provincia_operacao,0.023471,0.001136
1,eixos,0.016378,0.003407
8,intervalo_medio_os_hist,0.016367,0.002665
13,tire_size,0.014375,0.003765


## 10. Encerramento

Todas as figuras estão em `reports/figures/anual/` e todas as tabelas em `reports/tables/anual/`.
Este notebook é a fonte única de verdade da visão anual: reexecutá-lo regenera integralmente os artefatos
utilizados na apresentação.


In [13]:
print("Figuras:", len(os.listdir(FIG)), "arquivos em reports/figures/anual/")
print("Tabelas:", len(os.listdir(TAB)), "arquivos em reports/tables/anual/")
print("\nFiguras:"); [print("  ",x) for x in sorted(os.listdir(FIG))]
print("\nTabelas:"); [print("  ",x) for x in sorted(os.listdir(TAB))]


Figuras: 41 arquivos em reports/figures/anual/
Tabelas: 12 arquivos em reports/tables/anual/

Figuras:
   01_hist_y.png
   02_hist_y_log.png
   03_box_y.png
   04_evolucao_y.png
   05_heatmap_correlacao.png
   06_ranking_associacao.png
   10_scatter_n_os.png
   11_scatter_km.png
   12_grade_histogramas.png
   13_vif.png
   14_comparacao_modelos.png
   15_importancia_variaveis.png
   16_previsto_vs_observado.png
   cat_cod_montadora.png
   cat_flag_refrigerado.png
   cat_new_used_indicator.png
   cat_provincia_operacao.png
   cat_suspension_type.png
   cat_tire_size.png
   cat_unit_subtype.png
   cat_vmrs_predominante.png
   num_ano_modelo.png
   num_anos_desde_ultima_os.png
   num_comprimento.png
   num_custo_acum.png
   num_custo_ano_anterior.png
   num_custo_medio_os.png
   num_custo_preventivo_ano.png
   num_delta_km_medio_os.png
   num_eixos.png
   num_idade_carreta.png
   num_intervalo_medio_os_hist.png
   num_km_acumulado_defasado.png
   num_km_acumulado_fim_ano.png
   num_km_rod

[None, None, None, None, None, None, None, None, None, None, None, None]